In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


This notebook contains the code that estimates the subsequent impact on nutrients, enviornmenta indicators and cost foowing gram for gram replacements with alternative food groups. It further performs the uncertainty analysis on the standard error of each nutrient, environmental indicator and cost.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

data_path = Path("data")
results_path = Path("results")

In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from notebook_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import StrVector
from rpy2.robjects import r

In [ ]:
from substitution_simulations import replacement_sim
from demographic_conditions import *

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages("survey")
install.packages("readxl")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘minqa’, ‘numDeriv’, ‘mitools’, ‘RcppArmadillo’

trying URL 'https://cran.rstudio.com/src/contrib/minqa_1.2.8.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/numDeriv_2016.8-1.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/mitools_2.4.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RcppArmadillo_15.2.2-1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/survey_4.4-8.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpT3AsFp/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/readxl_1.4.5.tar.gz'
Content type 'application/x-gzip' length 1636512 bytes (1.6 MB)
downloaded 1.6 MB


The downloaded source packages are in
	‘/tmp/RtmpT3AsFp/downloaded_packages’


In [ ]:
survey = importr("survey")
readxl = importr("readxl")
dplyr = importr("dplyr")

Load the necessary data files

In [ ]:
df_baseline = pd.read_parquet(data_path / 'df_baseline.parquet')
nutrients_per_gram_sub = pd.read_parquet(data_path / 'nutrients_per_gram_sub.parquet')

env_columns = np.loadtxt(data_path / 'indicator_lists/env_columns.txt', dtype=str).tolist()
mean_env_columns = np.loadtxt(data_path / 'indicator_lists/mean_env_columns.txt', dtype=str).tolist()
nutrients = np.loadtxt(data_path / 'indicator_lists/nutrients.txt', dtype=str).tolist()
error_columns = np.loadtxt(data_path / 'indicator_lists/error_columns.txt', dtype=str).tolist()
all_meat_food_groups = np.loadtxt(data_path / 'indicator_lists/all_meat_food_groups.txt', dtype=str).tolist()


all_indicators = nutrients + env_columns

### Load mappings

In [ ]:
# Load dictionary mapping each error variable to the associated indicator
with open(data_path / 'mappings/standard_error_dict.json', 'r') as fp:
    standard_error_dict = json.load(fp)

# Map between each dairy food group that is being replaced and the corresponding substitute
with open(data_path / 'mappings/dairy_sub_mapping.json', 'r') as fp:
    dairy_sub_mapping = json.load(fp)

# Dictionary setting the table formatting parameters for the output tables
with open(data_path / 'mappings/nutrient_label_dict.json', 'r') as fp:
    nutrient_label_dict = json.load(fp)

# Set the parameters for each scenario
with open(data_path / 'mappings/scenario_dict.json', 'r') as fp:
    scenario_dict = json.load(fp)

# Convert the keys (scenario labels) to integers
scenario_dict = {int(k): v for k, v in scenario_dict.items()}


with open(data_path / 'mappings/standard_error_dict.json', 'r') as fp:
    standard_error_dict = json.load(fp)

# Scenario label in directory to the percent reduction in all meat consumption
ccc_scenario_mapping = {20: 22,
                        35: 23,
                        0: 34}

In [ ]:
demographic_columns = ['Sample Weight',
 'strata',
 'psu',
 'age',
 'Sex',
'white_scot',
 'white_OB',
 'asian',
 'white_oth',
 'oth_min_eth',
 'BMI',
 'SIMD1',
 'SIMD2',
 'SIMD3',
 'SIMD4',
 'SIMD5'
]

### Subgroup conditions

In [ ]:
overall_group = [("Overall", [ {'column': None, 'operator': None, 'value': None, 'boolean_operator': None}])]

subgroup_list_age = [

  ("16-24", [
                    {'column': 'age', 'operator': '>=', 'value': 16, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 24, 'boolean_operator': None}
                ]),

   ("25-34", [
      {'column': 'age', 'operator': '>=', 'value': 25, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 34, 'boolean_operator': None}
  ]),

   ("35-44", [
      {'column': 'age', 'operator': '>=', 'value': 35, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 44, 'boolean_operator': None}
  ]),

  ("45-54", [
      {'column': 'age', 'operator': '>=', 'value': 45, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 54, 'boolean_operator': None}
  ]),

  ("55-64", [
      {'column': 'age', 'operator': '>=', 'value': 55, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 64, 'boolean_operator': None}
  ]),

  ("65-74", [
      {'column': 'age', 'operator': '>=', 'value': 65, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 74, 'boolean_operator': None}
  ]),

  ("75+", [
      {'column': 'age', 'operator': '>=', 'value': 75, 'boolean_operator': None},

  ]),
            ]

subgroup_list_sex = [
  ("Female", [{'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("Male", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': None}])
]

subgroup_list_simd = [
  ("SIMD 1 (most deprived)", [{'column': 'SIMD1', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 2", [{'column': 'SIMD2', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 3", [{'column': 'SIMD3', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 4", [{'column': 'SIMD4', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 5 (least deprived)", [{'column': 'SIMD5', 'operator': '==', 'value': 1, 'boolean_operator': None}]),

]

subgroup_list_age_sex = [

                  ############## Male #####################

  ("M 16-24", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 16, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 24, 'boolean_operator': None}
                ]),

  ("M 25-34", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 25, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 34, 'boolean_operator': None}
                ]),

   ("M 35-44", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 35, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 44, 'boolean_operator': None}
  ]),

    ("M 45-54", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 45, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 54, 'boolean_operator': None}
  ]),

                  ("M 55-64", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 55, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 64, 'boolean_operator': None}
  ]),

  ("M 65+", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 65, 'boolean_operator': None},

  ]),

  #################### Female #########################

   ("F 16-24", [       {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 16, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 24, 'boolean_operator': None}
                ]),

  ("F 25-34", [ {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 25, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 34, 'boolean_operator': None}
                ]),

   ("F 35-44", [
       {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 35, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 44, 'boolean_operator': None}
  ]),

                  ("F 45-54", [ {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 45, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 54, 'boolean_operator': None}
  ]),

      ("F 55-64", [ {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 55, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 64, 'boolean_operator': None}
  ]),

  ("F 65+", [
      {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 65, 'boolean_operator': None},

  ])
            ]

all_conditions = [overall_group, subgroup_list_sex, subgroup_list_age, subgroup_list_simd] #, subgroup_list_age_sex]

### Run the substitutions for all scenarios

In [ ]:
# Set the number of reduction iterations to include in the analysis.Submitted results set max_seed=51, but has been set to 3 for demonstration purposes
max_seed = 51

for scenario, param_dict in scenario_dict.items():
    replacement_sim(base_path=results_path,
                    scenario=scenario,
                    df_baseline=df_baseline,
                    dairy_sub_mapping=dairy_sub_mapping,
                    nutrients_per_gram_sub = nutrients_per_gram_sub,
                    demographic_columns = demographic_columns,
                    ccc_scenario_mapping=ccc_scenario_mapping,
                    indicators = all_indicators,
                    nutrients = nutrients,
                    mean_env_columns = mean_env_columns,
                    error_columns = error_columns,
                    nutrient_label_dict=nutrient_label_dict,
                    standard_error_dict=standard_error_dict,
                    all_conditions = all_conditions,
                    max_seed = max_seed,
                    **param_dict
                    )

In [ ]:
from substitution_simulations import table_mean_values_subgroup

In [ ]:
def results_table_baseline(path_baseline: Path,
                          all_conditions: list,
                          df_baseline_total: pd.DataFrame,
                          nutrient_label_dict: dict,
                          nutrients: list,
                          mean_env_columns: list,
                          error_columns: list,
                          standard_error_dict: dict
                          ):

    nutrients_head = [f"{data['title']}, {data['units']}" for nutr, data in nutrient_label_dict.items() if not nutr.startswith('se_') and nutr in nutrients+mean_env_columns]
    standard_error_columns = [f"se_{nutr}" for nutr in nutrient_label_dict.keys() if not nutr.startswith('se') and nutr in nutrients+mean_env_columns]


    df_mean = pd.DataFrame(columns = nutrients_head+standard_error_columns)
    mean_filename = f"results_value_baseline.xlsx"

    for condition_list in all_conditions:
        survey_baseline, survey_mean, survey_change =  table_mean_values_subgroup(df_reduced_total=df_baseline_total,
                                                                                      subgroup_conditions = condition_list,
                                                                                      df_baseline_total = df_baseline_total,
                                                                                      nutrients = nutrients,
                                                                                      nutrient_label_dict = nutrient_label_dict,
                                                                                      mean_env_columns=mean_env_columns,
                                                                                      standard_error_dict= standard_error_dict
                                                                                    )

        #survey_change.rename(columns={nutr: f"{data['title']}, {data['units']}" for nutr, data in nutrient_label_dict.items() if not nutr.startswith('se_') and nutr in nutrients+mean_env_columns}, inplace=True )
        survey_mean.rename(columns={nutr: f"{data['title']}, {data['units']}" for nutr, data in nutrient_label_dict.items() if not nutr.startswith('se_') and nutr in nutrients+mean_env_columns}, inplace=True )
        for cond_label, _ in condition_list:
          df_mean.loc[cond_label, :] = survey_mean.loc[cond_label, :].round(2)

    df_mean.to_excel(path_baseline / mean_filename, index=True)

    return

In [ ]:
path_baseline = results_path / "Output/Scenarios/baseline/"

results_table_baseline(path_baseline=path_baseline,
                          all_conditions=all_conditions,
                          df_baseline_total=df_baseline,
                          nutrient_label_dict=nutrient_label_dict,
                          nutrients= nutrients,
                          mean_env_columns=mean_env_columns,
                          erro  ,                      r_columns = error_columns,
                          standard_error_dict = standard_error_dict
                          )